#### Decision API 2차 배치 검수

`ner_human_review.review_method = 'LLM검수'` 대상만 PostgreSQL에서 스트리밍 추출하고, 언론사별로 태그 오류와 태깅 병합 필요 여부를 판단한다.

- API 실행은 3번 셀에서 `CURRENT_PRESS`로 지정한 언론사 1곳씩 진행한다.
- 안전 기준 초과 또는 10건 테스트 실패 시 전체 실행은 자동 중단한다.
- 완료된 식별자는 언론사별 `results.jsonl`에서 읽어 재호출하지 않는다.

In [2]:
import os
import sys
from datetime import datetime
from itertools import islice
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path(r'C:\work\ner')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import get_postgre_connect
from src.ner_review.decision.decision import DECISION_MODEL
from src.ner_review.decision.batch import (
    DEFAULT_BATCH_SIZE,
    DEFAULT_MAX_WORKERS,
    DEFAULT_MAX_RETRIES,
    DEFAULT_REQUEST_TIMEOUT_SECONDS,
    build_decision_input,
    estimate_usage,
    run_press_list,
    stream_review_targets,
)

DATA_VERSION = None # os.getenv('DATA_VERSION') or None
MODEL_OVERRIDE = None  # Decisions API 공식 지원 모델 외 값은 사용하지 않음
MODEL = MODEL_OVERRIDE or DECISION_MODEL
if MODEL != 'gpt-6-luna':
    raise ValueError(f'Decisions API 지원 모델은 gpt-6-luna입니다: {MODEL}')
RUN_API = True  # 3번 셀에서 CURRENT_PRESS 언론사 1곳을 실행
PRESS_FILTER = None  # 예: ['강원일보']
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'decision_2' / f'run_{datetime.now():%Y%m%d_%H%M%S}'

# API 제한과 비용에 맞춰 조정하는 설정
BATCH_SIZE = DEFAULT_BATCH_SIZE
MAX_WORKERS = DEFAULT_MAX_WORKERS
MAX_RETRIES = DEFAULT_MAX_RETRIES
BACKOFF_SECONDS = 2.0
MAX_BACKOFF_SECONDS = 60.0
REQUEST_TIMEOUT_SECONDS = DEFAULT_REQUEST_TIMEOUT_SECONDS
MAX_INPUT_CHARS = 120_000
SAMPLE_SIZE = 100
CHARS_PER_TOKEN = 4.0
ESTIMATED_OUTPUT_TOKENS_PER_REQUEST = 300
INPUT_PRICE_PER_1M = None   # 모델 가격표 확인 후 숫자로 입력
OUTPUT_PRICE_PER_1M = None  # 모델 가격표 확인 후 숫자로 입력

print('output_root:', OUTPUT_ROOT)
print('model:', MODEL, '| run_api:', RUN_API)

output_root: C:\work\ner\outputs\decision_2\run_20261120_014739
model: gpt-6-luna | run_api: True


##### 1. 대상 언론사와 소규모 입력 확인

In [3]:
def list_press_names(data_version=None):
    where = "WHERE review_method = 'LLM검수'"
    params = []
    if data_version:
        where += ' AND data_version = %s'
        params.append(data_version)
    with get_postgre_connect() as connection:
        rows = connection.execute(
            f'SELECT DISTINCT press FROM ner_human_review {where} ORDER BY press',
            params,
        ).fetchall()
    return [row[0] for row in rows]

press_list = list_press_names(DATA_VERSION)
if PRESS_FILTER:
    press_list = [press for press in press_list if press in PRESS_FILTER]
print('presses:', press_list)

preview_rows = []
for press in press_list[:3]:
    target = next(stream_review_targets(data_version=DATA_VERSION, press=press), None)
    if target:
        preview_rows.append({
            'press': press,
            'data_version': target['data_version'],
            'doc_id': target['doc_id'],
            'sentence_id': target['sentence_id'],
            'original_form': target['original_form'],
            'entity_count': len(target['entities']),
            'active_rule_count': len(target['sentence_rule_evidence']),
        })
display(pd.DataFrame(preview_rows))

presses: ['강원일보', '경기일보', '경향신문', '노컷뉴스', '뉴스1', '뉴스핌', '뉴시스', '디지털타임스', '마이데일리', '부산일보', '서울신문', '세계일보', '스포츠경향', '스포츠투데이', '아시아투데이', '이데일리', '이투데이', '일간투데이', '충청일보']


,press,data_version,doc_id,sentence_id,original_form,entity_count,active_rule_count
0,강원일보,20261117_192958_강원일보,NLRW2600000001.10006,NLRW2600000001.10006.6.2,<2022년: DT> <20대 대선: EV>의 <강원: LC>지역 <1일차: DT>...,1,1
1,경기일보,20261117_193354_경기일보,NLRW2600000002.10002,NLRW2600000002.10002.1.1,"<박남춘: PS> “<이재명: PS> 원팀, 정권교체”…<1년: DT>만 SNS 글...",2,2
2,경향신문,20261117_193858_경향신문,NWRW2600000001.10004,NWRW2600000001.10004.5.1,앞서 <법사위: OG>는 <윤석열: PS> 대통령의 구속취소 결정에 대한 <이날: ...,1,1


##### 2. 언론사별 요청 수·토큰·비용 추정

전체 입력은 메모리에 모으지 않고, 언론사별 generator를 한 번 순회한다. 토큰은 문자 수 기반 보수적 추정이며 실제 사용량은 결과 JSONL의 `api.usage`를 확인한다.

In [3]:
usage_rows = []
for press in press_list:
    target_iter = stream_review_targets(data_version=DATA_VERSION, press=press)
    sample_targets = list(islice(target_iter, SAMPLE_SIZE))
    usage = estimate_usage(sample_targets, sample_size=SAMPLE_SIZE, chars_per_token=CHARS_PER_TOKEN)
    remaining_count = sum(1 for _ in target_iter)
    estimated_request_count = len(sample_targets) + remaining_count
    estimated_input_tokens = usage['estimated_input_tokens']
    estimated_input_tokens_total = estimated_request_count * estimated_input_tokens
    estimated_output_tokens_total = estimated_request_count * ESTIMATED_OUTPUT_TOKENS_PER_REQUEST
    estimated_cost = None
    if INPUT_PRICE_PER_1M is not None and OUTPUT_PRICE_PER_1M is not None:
        estimated_cost = (estimated_input_tokens_total / 1_000_000 * INPUT_PRICE_PER_1M) + (estimated_output_tokens_total / 1_000_000 * OUTPUT_PRICE_PER_1M)
    usage_rows.append({
        'press': press,
        'sample_size': usage['sample_size'],
        'estimated_request_count': estimated_request_count,
        'avg_input_chars': usage['avg_input_chars'],
        'max_input_chars': usage.get('max_input_chars', 0),
        'estimated_input_tokens_per_request': estimated_input_tokens,
        'estimated_input_tokens_total': estimated_input_tokens_total,
        'estimated_output_tokens_total': estimated_output_tokens_total,
        'estimated_cost': estimated_cost,
    })
usage_frame = pd.DataFrame(usage_rows)
display(usage_frame)
print('주의: 정확한 요청 수/사용 토큰은 각 press/summary.json에 누적 기록됩니다.')

,press,sample_size,estimated_request_count,avg_input_chars,max_input_chars,estimated_input_tokens_per_request,estimated_input_tokens_total,estimated_output_tokens_total,estimated_cost
0,강원일보,100,2177,504.8,765,126,274302,653100,None
1,경기일보,100,2706,537.7,863,134,362604,811800,None
2,경향신문,100,6372,525.1,797,131,834732,1911600,None
3,노컷뉴스,100,9076,551.5,1183,138,1252488,2722800,None
4,뉴스1,100,28855,545.4,840,136,3924280,8656500,None
5,뉴스핌,100,14403,552.1,1024,138,1987614,4320900,None
6,뉴시스,100,22106,534.3,1207,134,2962204,6631800,None
7,디지털타임스,100,5404,540.0,751,135,729540,1621200,None
8,마이데일리,100,7706,519.0,745,130,1001780,2311800,None
9,부산일보,100,6347,543.5,952,136,863192,1904100,None


주의: 정확한 요청 수/사용 토큰은 각 press/summary.json에 누적 기록됩니다.


##### 3. 언론사별 순차 배치 실행

`MAX_WORKERS`만 한 언론사 내부에서 병렬 처리하고, 언론사 자체는 순차 처리한다. 배치가 끝날 때마다 `results.jsonl`, `failures.jsonl`, `summary.json`, `execution.log.jsonl`을 저장한다.

In [11]:
press_listt = press_list[:1]
press_listt

['강원일보']

In [ ]:
# CURRENT_PRESS = '경향신문'  # 실행할 언론사를 1곳씩 지정

def get_press_data_version(press):
    with get_postgre_connect() as connection:
        rows = connection.execute(
            "SELECT DISTINCT data_version FROM ner_human_review "
            "WHERE review_method = 'LLM검수' AND press = %s ORDER BY data_version DESC",
            [press],
        ).fetchall()
    data_versions = [row[0] for row in rows]
    if not data_versions:
        raise ValueError(f'ner_human_review에 데이터가 없습니다: {press}')
    if DATA_VERSION and DATA_VERSION not in data_versions:
        raise ValueError(f'{press}에 DATA_VERSION이 없습니다: {DATA_VERSION}')
    if DATA_VERSION:
        return DATA_VERSION
    if len(data_versions) > 1:
        raise ValueError(f'{press}에 여러 data_version이 있습니다: {data_versions}')
    return data_versions[0]

for current_press in press_listt:
    CURRENT_PRESS = current_press

    PRESS_DATA_VERSION = get_press_data_version(CURRENT_PRESS)
    print(f'실행 언론사: {CURRENT_PRESS} | data_version: {PRESS_DATA_VERSION}')

    if RUN_API:
        summaries = run_press_list(
            [CURRENT_PRESS],
            OUTPUT_ROOT,
            data_version=PRESS_DATA_VERSION,
            model=MODEL,
            batch_size=BATCH_SIZE,
            max_workers=MAX_WORKERS,
            max_retries=MAX_RETRIES,
            backoff_seconds=BACKOFF_SECONDS,
            max_backoff_seconds=MAX_BACKOFF_SECONDS,
            request_timeout_seconds=REQUEST_TIMEOUT_SECONDS,
            max_input_chars=MAX_INPUT_CHARS,
            progress=lambda event: print(event),
        )
        display(pd.DataFrame(summaries))
    else:
        print('RUN_API=False: 실제 Decision API 호출을 실행하지 않았습니다.')

##### 4. 실행 현황과 결과 확인

In [5]:
def load_summaries(output_root):
    paths = sorted(Path(output_root).glob('press/*/summary.json'))
    rows = []
    for path in paths:
        rows.append(__import__('json').loads(path.read_text(encoding='utf-8')).copy())
        rows[-1]['summary_path'] = str(path)
    return pd.DataFrame(rows)

summary_frame = load_summaries(OUTPUT_ROOT)
if CURRENT_PRESS:
    summary_frame = summary_frame[summary_frame['press'] == CURRENT_PRESS]
display(summary_frame)

press_dir = Path(OUTPUT_ROOT) / 'press' / CURRENT_PRESS
result_path = press_dir / 'results.jsonl'
failure_path = press_dir / 'failures.jsonl'
if result_path.exists():
    result_frame = pd.read_json(result_path, lines=True)
    display(result_frame)
if failure_path.exists():
    failure_frame = pd.read_json(failure_path, lines=True)
    display(failure_frame)
print('결과 루트:', OUTPUT_ROOT)

,schema_version,run_id,press,data_version,model,prompt_version,started_at,status,batch_size,max_workers,...,skipped,pending,input_tokens,output_tokens,request_count,retried_requests,retry_targets,unprocessed,finished_at,summary_path
0,ner-decision-v2,run_20261120_014739,경기일보,20261117_193354_경기일보,gpt-6-luna,decision-v2,2026-11-19T16:48:08.148456+00:00,completed,20,4,...,0,0,22490855,0,2706,0,0,0,2026-11-19T16:51:55.069643+00:00,C:\work\ner\outputs\decision_2\run_20261120_01...


,schema_version,status,run_id,executed_at,model,prompt_version,data_version,press,doc_id,sentence_id,original_form,tagging_error,tagging_merge,source,api
0,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:48:09.604858+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.10002,NLRW2600000002.10002.1.1,"<박남춘: PS> “<이재명: PS> 원팀, 정권교체”…<1년: DT>만 SNS 글...","{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
1,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:48:09.591388+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.1003,NLRW2600000002.1003.5.2,<권: PS> 대표는 “출범한 지 <5년: DT> 동안 구속영장과 체포영장 한 번 ...,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
2,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:48:09.594182+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.10040,NLRW2600000002.10040.10.1,<시: OG>는 <다음 달: DT>부터 본격적인 입소자 신청 접수를 시작할 예정이다.,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
3,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:48:09.642017+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.10045,NLRW2600000002.10045.1.1,"<경찰: OG>, <대선: REVIEW_EV> 후보 경호 시작…<6·3 대선: EV...","{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
4,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:48:09.889264+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.1007,NLRW2600000002.1007.3.2,<올해: DT>부터는 횟수 제한 없이 1인당 최대 10만원까지 어학시험 및 국가자격...,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2701,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:51:54.761284+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.9990,NLRW2600000002.9990.8.1,<이: PS> 후보는 <이날: DT> 수락연설에서 “<23년 전: DT> <오늘: ...,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
2702,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:51:54.741972+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.9991,NLRW2600000002.9991.1.1,<이재명: PS> “대통령 제1과제 국민통합 책임 확실하게 완수” [<6·3 대선:...,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
2703,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:51:54.760290+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.9996,NLRW2600000002.9996.1.1,"<이재명: PS> “압도적 지지, 압도적 기대라 생각…무겁게 받아들일 것” [<6·...","{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 'mer...",{'existing_entities': [{'entity_id': 'NLRW2600...,"{'response_id': None, 'attempts': 1, 'input_ch..."
2704,ner-decision-v2,completed,run_20261120_014739,2026-11-19 16:51:54.997929+00:00,gpt-6-luna,decision-v2,20261117_193354_경기일보,경기일보,NLRW2600000002.9998,NLRW2600000002.9998.1.1,막 내린 <김동연: PS> <경기도지사: OG>의 도전…역부족 속에도 부각된 존재감...,"{'is_error': False, 'error_type': None, 'targe...","{'needed': False, 'merge_entity_ids': [], 

""


결과 루트: C:\work\ner\outputs\decision_2\run_20261120_014739


##### 실행 순서(언론사별 수동 실행)

1. 위 설정 셀과 1·2번 셀을 먼저 실행한다.
2. 3번 셀의 `CURRENT_PRESS`에 실행할 언론사 1곳을 지정하고 3번 셀만 실행한다.
3. 실행이 끝나면 4번 셀을 실행해 해당 언론사의 summary와 결과를 확인한다.
4. 다음 언론사를 실행할 때는 `CURRENT_PRESS`만 바꾼 뒤 3번과 4번 셀을 반복 실행한다.
5. 모든 언론사는 같은 `OUTPUT_ROOT` 아래 언론사별 폴더에 저장된다.